<!-- Cell 0 -->
# Policy Intelligence POC — Run One Policy

Flow: **config → LLM → run → archive report → offline pack** (no `*_FULL.zip`).  
**CITI / workstation:** put the personal trading policy under `policies/`, set `POLICY_PATH` / `POLICY_ID` in Cell 4, then **Run All once**.

Outputs:
- Pipeline demo: `policy_runs/<POLICY_ID>/<RUN_ID>/demo/index.html`
- Offline pack (main + role views): `notebooks/offline_p2b/index.html` (section 3b)
- Offline zip for handoff: `notebooks/offline_p2b_OFFLINE_DEMO.zip` (section 3b or `pack_offline_demo.bat`)

**Offline views (one pack):**
- **Main** — `index.html` (BPMN workspace). Intelligence lens defaults to **All roles (main view)**.
- **Policy Follower / Compliance Officer / Policy Owner** — opens `role_intelligence.html` with workflows/BPMN from the same pack.

Pass F embeds `role_relevance` + `role_workflow_map` into `data.js` when Cell 8 produced a run.  
Home setup: `docs/home_setup.md`.


<!-- Cell 1 -->
## 0. Home only — install packages (skip on CITI JupyterHub)

Run once on a blank home env, then **Kernel → Restart**, then continue.

In [1]:
# Cell 2
# HOME ONLY — do not run on Hub (%pip / ad-hoc installs are blocked there).
import sys
import subprocess

PACKAGES = [
    "pandas", "pydantic", "python-docx", "beautifulsoup4", "lxml", "jinja2",
    "networkx", "pytest", "ipython", "PyYAML", "pypdf", "openai",
]
cmd = [sys.executable, "-m", "pip", "install", "--upgrade", *PACKAGES]
print("Running:", " ".join(cmd))
subprocess.check_call(cmd)
print("\nOK — restart the kernel, then Run All from Bootstrap.")

Running: c:\Python311\python.exe -m pip install --upgrade pandas pydantic python-docx beautifulsoup4 lxml jinja2 networkx pytest ipython PyYAML pypdf openai

OK — restart the kernel, then Run All from Bootstrap.


<!-- Cell 3 -->
## 1. Bootstrap + config

1. Put your policy under `policies/` (pdf / docx / html).
2. Set `POLICY_ID` and `POLICY_PATH` below.
3. Optional: `.env` with `OPENAI_API_KEY` and `POLICY_LLM_PROVIDER=openai`.

In [2]:
# Cell 4
import os
import sys
from pathlib import Path

# --- Project root on sys.path (must come before policy_knowledge imports) ---
ROOT = Path.cwd()
if (ROOT / "src").exists():
    sys.path.insert(0, str(ROOT / "src"))
elif (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
    sys.path.insert(0, str(ROOT / "src"))

# Notebook helpers (offline p2b pack builder)
if str(ROOT / "notebooks") not in sys.path:
    sys.path.insert(0, str(ROOT / "notebooks"))

from policy_knowledge.config import load_config, TransferPolicy


def load_dotenv(path: Path) -> None:
    """Load KEY=VALUE from .env into os.environ (does not override existing)."""
    if not path.is_file():
        return
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, value = line.partition("=")
        key, value = key.strip(), value.strip().strip('"').strip("'")
        if key and key not in os.environ:
            os.environ[key] = value


load_dotenv(ROOT / ".env")
config = load_config(ROOT / "config" / "app.example.yaml")

# # --- Edit these for your run ---
# POLICY_ID = os.environ.get("POLICY_WORKSPACE_ID") or "PTIP_BR_001"

# # Point at your file under policies/. Env POLICY_PATH overrides when set.
# POLICY_PATH = Path(
#     os.environ["POLICY_PATH"]
#     if os.environ.get("POLICY_PATH")
#     else ROOT / "policies" / "PTIP_black_rosk-21-2026.pdf"
# )


POLICY_ID = "PTIP_BR_001"
POLICY_PATH = ROOT / "policies" / "PTIP_black_rock-21-2026.pdf"

# data.js always gets full parsed text (feeds demo/index.html).
# workstation_transfer_approved only gates creating a shareable DEMO ZIP.
transfer_policy = TransferPolicy(
    include_policy_excerpts_in_demo=True,
    workstation_transfer_approved=False,
)

print("ROOT", ROOT)
print("POLICY_ID", POLICY_ID)
print("POLICY_PATH", POLICY_PATH, "exists=", POLICY_PATH.exists())
print("OPENAI_API_KEY set=", bool(os.environ.get("OPENAI_API_KEY")))

ROOT d:\PORTFOLIO\CITI
POLICY_ID PTIP_BR_001
POLICY_PATH d:\PORTFOLIO\CITI\policies\PTIP_black_rock-21-2026.pdf exists= True
OPENAI_API_KEY set= True


<!-- Cell 5 -->
## 2. LLM provider

`mock` = offline synthetic · `openai` = home (needs key) · `guardrail` = CITI Hub

In [3]:
# Cell 6
from policy_knowledge.intelligence.guardrail_client import GuardRailUnavailable
from policy_knowledge.intelligence.llm_factory import create_llm_client, llm_runtime_status
from policy_knowledge.intelligence.openai_client import OpenAIUnavailable
from policy_knowledge.intelligence.synthetic_mock import SyntheticMockLLM

# Prefer .env POLICY_LLM_PROVIDER; otherwise yaml / mock.
# Force OpenAI: LLM_PROVIDER = "openai"
LLM_PROVIDER = os.environ.get("POLICY_LLM_PROVIDER", config.llm.provider or "mock")
config.llm.provider = LLM_PROVIDER

try:
    llm = create_llm_client(config, require_live=(LLM_PROVIDER == "guardrail"))
except (OpenAIUnavailable, GuardRailUnavailable) as exc:
    print(exc)
    llm = SyntheticMockLLM()
    config.llm.provider = "mock"

status = llm_runtime_status(llm)
print(status["llm_status"], type(llm).__name__, "provider=", config.llm.provider)
if LLM_PROVIDER == "openai" and status["llm_status"] != "OPENAI_CONNECTED":
    print("Hint: set OPENAI_API_KEY in .env — see docs/home_setup.md")

OPENAI_CONNECTED OpenAIAdapter provider= openai


<!-- Cell 7 -->
## 3. Run pipeline + archive

Writes `demo/data.js` (full text) + OKF/BPMN/matrices + `role_relevance.json`.  
**No `*_FULL.zip`** — offline handoff is section **3b** → `offline_p2b_OFFLINE_DEMO.zip`. DEMO ZIP only if `workstation_transfer_approved=True`.

In [5]:
# Cell 8
import json
print(transfer_policy)
from policy_knowledge.packaging.package_policy import archive_policy_run
from policy_knowledge.pipeline import run_policy_demo

# Ingest → extract → export OKF/BPMN/demo (data.js populated) → QA
result = run_policy_demo(
    POLICY_PATH,
    policy_id=POLICY_ID,
    config=config,
    llm=llm,
    transfer_policy=transfer_policy,
    overwrite=True,
)

demo_html = result.run_dir / "demo" / "index.html"
data_js = result.run_dir / "demo" / "data.js"
role_json = result.run_dir / "outputs" / "canonical" / "role_relevance.json"
print("run_dir", result.run_dir)
print("open HTML", demo_html)
print("data.js bytes", data_js.stat().st_size if data_js.exists() else 0)
print("role_relevance", role_json if role_json.exists() else "MISSING")
print("validation", json.dumps(result.validation, indent=2)[:2000])

# No *_FULL.zip (use section 3b offline_p2b zip). DEMO zip only if transfer approved.
archives = archive_policy_run(result.run_dir, transfer_policy=transfer_policy)
print("full_zip", archives.full_zip)  # always None
print("demo_zip", archives.demo_zip)
print("report", archives.report_path)

TransferPolicy(workstation_transfer_approved=False, include_source_in_full=False, include_intermediate_in_full=False, include_policy_excerpts_in_demo=True)
run_dir policy_runs\PTIP_BR_001\20261120T022301Z-POC
open HTML policy_runs\PTIP_BR_001\20261120T022301Z-POC\demo\index.html
data.js bytes 65208
validation {
  "knowledge": {
    "ok": true,
    "errors": [],
    "warnings": []
  },
  "okf": {
    "ok": true,
    "errors": [],
    "warnings": []
  },
  "bpmn": {
    "ok": true,
    "errors": [],
    "warnings": []
  },
  "offline_demo": {
    "ok": true,
    "errors": [],
    "warnings": [
      "interactive demo not ready (vendor assets missing or stubs)",
      "BROWSER_TEST_NOT_RUN"
    ]
  },
  "quality": {
    "schema": {
      "ok": true,
      "errors": [],
      "warnings": []
    },
    "graph": {
      "ok": true,
      "errors": [],
      "warnings": []
    },
    "finding_count": 16,
    "requirement_count": 10,
    "edge_count": 29,
    "inferred_edge_count": 0
  }
}
ful

<!-- Cell 9 -->
## 3b. Offline pack (main + role views) + zip

Packages `policies2BPMN/outputs` into `notebooks/offline_p2b/`:
- **Main view** — `index.html`
- **Role views** — `role_intelligence.html` (Policy Follower / Compliance / Policy Owner)
- Embedded `data.js` with workflows, BPMN, Pass F lenses, and `role_workflow_map`

Also writes **`notebooks/offline_p2b_OFFLINE_DEMO.zip`** for offline handoff (or run `pack_offline_demo.bat` at repo root).

Open the printed `file://` URI. Does **not** modify `policies2BPMN/`.  
Requires `policies2BPMN/viewer/node_modules/bpmn-js` (`npm install` once in `policies2BPMN/viewer` if missing).


In [ ]:
# Cell 10
import importlib
import json
from pathlib import Path

import build_offline_p2b

importlib.reload(build_offline_p2b)  # pick up disk fixes without Kernel Restart
from build_offline_p2b import build_offline_p2b, zip_offline_pack

# Prefer the pipeline run from Cell 8 when available (Pass F lenses + role map).
run_dir = locals().get("result") and getattr(result, "run_dir", None)
build_kwargs = {}
if run_dir is not None:
    canonical = Path(run_dir) / "outputs" / "canonical"
    if (canonical / "role_relevance.json").is_file():
        build_kwargs["canonical_run_dir"] = Path(run_dir)

out = build_offline_p2b(ROOT, ROOT / "notebooks" / "offline_p2b", **build_kwargs)
zip_path = zip_offline_pack(out, ROOT / "notebooks" / "offline_p2b_OFFLINE_DEMO.zip")
html = out / "index.html"
manifest = json.loads((out / "OFFLINE_MANIFEST.json").read_text(encoding="utf-8"))
print("offline pack", out)
print("open main view", html.resolve().as_uri())
print("role views", manifest.get("role_views"))
print("workflows", len((out / "data.js").read_text(encoding="utf-8").split('"workflow_id"')) - 1)
print("role_relevance_embedded", manifest.get("role_relevance_embedded"))
print("role_workflow_map_embedded", manifest.get("role_workflow_map_embedded"))
print("offline_demo_zip", zip_path)
print("handoff: unzip offline_p2b_OFFLINE_DEMO.zip → open offline_p2b/index.html")


offline pack D:\PORTFOLIO\CITI\notebooks\offline_p2b
open HTML file:///D:/PORTFOLIO/CITI/notebooks/offline_p2b/index.html
workflows 32


<!-- Cell 11 -->
## 4. Optional — run tests

In [ ]:
# Cell 12
import pytest

pytest.main(["-q", str(ROOT / "tests")])

In [ ]:
# Cell 13